# 📝 지식그래프 구축 과제 LV1(기초): 공공데이터 적재

> 이 단원의 새 기술을 **하나씩** 확인합니다. 인코딩 확인, 제약조건, 적재 전 점검, `LOAD CSV` 형변환 적재, 멱등 재실행, `$파라미터` 조회, `apoc.meta.stats`, `UNWIND` 소량 적재, 배치 트랜잭션, 날짜를 노드로 올려 잇기.

## 풀이 방법
1. 맨 위 **준비 셀들**(연결·초기화·인코딩 변환·복사)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: `data/seoul_metro_transfer_cp949.csv`. 서울교통공사가 공개한 **환승역별 요일 환승인원**입니다. 포털에서 받은 바이트 그대로라 인코딩이 `CP949` 입니다.
- 연결은 준비 셀의 `run_cypher("쿼리", 파라미터=값)` 헬퍼로 합니다.

화이팅!

> **데이터 출처**: 아래 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다. 교육용으로 지어낸 값이 없습니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | CC0 |
> | 서울교통공사 역간거리·소요시간 (`seoul_metro_stations_cp949.csv`) | 서울 열린데이터광장 OA-12034 | 공공누리 1유형(출처표시) |
> | 서울교통공사 환승역 환승인원 (`seoul_metro_transfer_cp949.csv`) | 서울 열린데이터광장 OA-12033 | 공공누리 1유형(출처표시) |
>
> 의료 그래프는 **2016년에 정리된 자료**입니다. 그래서 "이 약이 이 병에 쓰인다고 **문헌에 정리돼 있다**"까지가 이 데이터가 말하는 것이고, "효능이 입증됐다"는 아닙니다. 지식그래프를 다룰 때 이 구분을 놓치면 안 됩니다.
>
> 지하철 CSV 두 개는 포털에서 받은 **바이트 그대로**라 인코딩이 `CP949` 입니다. UTF-8 로 읽으면 글자가 깨집니다.

아래 준비 셀들을 위에서부터 실행하세요. Neo4j 는 반드시 **실습 전용 DB**에 연결하세요.

이 과제는 **Neo4j Desktop** 에서 풉니다. 2번 문제가 `IS NODE KEY` 를 요구하고 채점도 제약의 종류를 보기 때문에, 그 문법이 막힌 에디션에서는 통과할 수 없습니다. Desktop 에는 필요한 라이선스가 딸려 옵니다.

파일을 못 읽거나 APOC 를 못 찾는 에러가 나면 `환경_구축_가이드.md` 의 오류 표를 보세요.

> 변환 셀이 `seoul_metro_transfer_cp949.csv` 를 UTF-8 로 바꿔 `seoul_metro_transfer_utf8.csv` 로 저장합니다. `LOAD CSV` 는 UTF-8 만 읽기 때문입니다(교안_03 1절). 1번 문제에서는 **원본 CP949 파일**을 직접 다룹니다.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 공공데이터 CSV 를 UTF-8 로 바꿔 둡니다: 이 셀은 실행만 하세요.
# LOAD CSV 는 인코딩을 지정할 수 없어 UTF-8 파일만 읽습니다(교안_03 1절).
from pathlib import Path

_src = Path('data') if Path('data').exists() else Path('../data')
_text = (_src / 'seoul_metro_transfer_cp949.csv').read_text(encoding='cp949')
(_src / 'seoul_metro_transfer_utf8.csv').write_text(_text, encoding='utf-8')
print('UTF-8 로 저장:', 'seoul_metro_transfer_utf8.csv')


In [ ]:
# [제공 코드] 데이터 파일을 Neo4j import 폴더로 복사: 이 셀은 실행만 하세요.
# LOAD CSV·apoc.load.json 은 보안상 서버의 import 폴더 안 파일만 읽습니다.
# - .env 에 NEO4J_IMPORT_DIR 이 있으면 data/ 의 파일을 자동 복사합니다.
# - 없으면(수동 복사한 경우) 그대로 넘어갑니다. Neo4j Desktop 은 인스턴스 메뉴의
#   "Open folder > Import" 로 폴더를 열어 data/ 의 파일을 직접 복사해 두세요.
import shutil
from pathlib import Path

_IMPORT_DIR = os.getenv("NEO4J_IMPORT_DIR", "")
_SRC_DIR = Path("data") if Path("data").exists() else Path("../data")
if _IMPORT_DIR:
    # csv 와 json 만 복사합니다. 다음 준비 셀이 이 파일들을 file:/// 로 읽습니다
    for f in sorted(_SRC_DIR.glob("*.csv")) + sorted(_SRC_DIR.glob("*.json")):
        shutil.copy(f, Path(_IMPORT_DIR) / f.name)
        print("복사:", f.name)
else:
    print("NEO4J_IMPORT_DIR 미설정: data/ 의 파일을 import 폴더에 직접 복사했는지 확인하세요.")

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 적재할 CSV 의 앞부분을 미리 봅니다(값이 전부 문자열인 것도 확인).

In [ ]:
# [제공 코드] 적재 전 CSV 미리보기
for p in run_cypher("LOAD CSV WITH HEADERS FROM 'file:///seoul_metro_transfer_utf8.csv' AS row "
                    "RETURN row LIMIT 3"):
    print(p['row'])

## 1. CP949 파일의 머리글 확인하기
**배경**: 국내 공공데이터는 `CP949` 인코딩이 흔합니다. UTF-8 로 읽으면 글자가 깨지거나 에러가 납니다. 적재 전에 **인코딩을 지정해 파이썬으로 열어** 머리글부터 확인합니다.

**요구사항**:
- 파일 경로를 받아 **머리글(칸 이름) 목록**을 리스트로 돌려주는 함수 **`read_headers(path)`** 를 만드세요. 인코딩을 **`cp949`** 로 지정해 읽습니다.
- 그 함수로 `data/seoul_metro_transfer_cp949.csv` 의 머리글을 **`headers`** 에 담으세요.
- `pandas` 로 읽는 것을 권합니다(`pd.read_csv(path, encoding='cp949')` 의 `columns`). 다른 방법으로 읽어도 됩니다. 채점은 **돌려준 목록**만 봅니다.

**예시**: `headers` 는 `['연번', '역명', '평일(일평균)', '토요일', '일요일']` 입니다(파일 머리글 순서 그대로).

> 채점 셀은 같은 함수를 **머리글이 다른 CP949 파일**(`seoul_metro_stations_cp949.csv`)로 한 번 더 호출합니다. 파일 이름을 함수 안에 박지 말고 `path` 로 받으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 경로를 받아 인코딩을 지정해 읽고, 표의 열 이름 목록을 돌려주는 함수를 만든다.

세부구현:
1. pandas 를 import 한다.
2. read_headers(path) 안에서 인코딩을 cp949 로 지정해 표를 읽는다.
3. 읽어 온 표의 열 이름은 columns 에 들어 있다.
4. 그것을 리스트로 만들어 돌려준다. 그 함수를 불러 headers 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 비교 전에 형부터 본다. pandas 의 columns 를 그대로 돌려주면 값이 다 맞아도
# 리스트 비교가 배열 비교가 되어 엉뚱한 에러로 죽는다
assert isinstance(headers, list), \
    f'리스트로 돌려주세요(현재 {type(headers).__name__}). pandas 의 columns 는 list(...) 로 감싸야 합니다'
assert headers == ['연번', '역명', '평일(일평균)', '토요일', '일요일'], \
    f'머리글 다섯 개를 파일 순서대로 담아야 합니다(현재 {headers}). encoding=\'cp949\' 를 썼는지 확인하세요'
# 머리글이 다른 파일로 같은 함수를 한 번 더 부른다. 지문 값을 그대로 적은 답안은 여기서 걸린다
_dir = Path('data') if Path('data').exists() else Path('../data')
other = read_headers(_dir / 'seoul_metro_stations_cp949.csv')
assert isinstance(other, list), \
    f'이쪽도 리스트여야 합니다(현재 {type(other).__name__}). list(...) 로 감싸세요'
assert other == ['연번', '호선', '역명', '소요시간', '역간거리(km)', '호선별누계(km)'], \
    f'같은 함수가 다른 CP949 파일의 머리글도 읽어야 합니다(현재 {other}). 파일 이름을 함수 안에 박지 말고 path 로 받으세요'
print('✅ 통과!')

## 2. 키 제약조건 걸기
**배경**: 같은 역이 중복 적재되지 않도록, 적재 **전에** 역명에 **키 제약**을 겁니다(교안_01 에서 정한 순서 그대로입니다). 키 제약은 **겹치지 않을 것**과 **비어 있지 않을 것**을 함께 요구하고, 제약이 만드는 인덱스가 `MERGE` 를 빠르게 만들기도 합니다.

**요구사항**:
- 레이블 **`Station`** 의 **`name`** 속성에 **키 제약**(`IS NODE KEY`)을 이름 **`station_name`** 으로 만드세요(`IF NOT EXISTS` 포함).

**예시**: 만든 뒤 `SHOW CONSTRAINTS` 에 이름이 `station_name`, 대상이 `['Station']`·`['name']`, 종류가 `NODE_KEY` 인 제약이 보입니다.

<details><summary>힌트</summary>

```text
접근방법:
- CREATE CONSTRAINT 문 하나를 실행한다. 이름·레이블·속성·종류를 지문 그대로 맞춘다.

세부구현:
1. CREATE CONSTRAINT 뒤에 제약 이름을 적고 IF NOT EXISTS 를 붙인다.
2. FOR 절에 대상 노드 패턴을, REQUIRE 절에 키가 될 속성과 IS NODE KEY 를 적는다.
3. run_cypher 로 실행한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
cons = run_cypher("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties, type "
                  "RETURN name, labelsOrTypes, properties, type")
found = [c for c in cons if c['name'] == 'station_name']
assert found, f'station_name 이라는 이름의 제약이 필요합니다(현재 {[c["name"] for c in cons]})'
assert found[0]['labelsOrTypes'] == ['Station'], \
    f'대상 레이블은 Station 이어야 합니다(현재 {found[0]["labelsOrTypes"]})'
assert found[0]['properties'] == ['name'], \
    f'대상 속성은 name 이어야 합니다(현재 {found[0]["properties"]})'
assert found[0]['type'] == 'NODE_KEY', \
    f'키 제약(IS NODE KEY)이어야 합니다(현재 {found[0]["type"]})'
print('✅ 통과!')

## 3. 적재 전에 오염 행 세기
**배경**: "몇 개가 버려지는지 모르는 채로 적재하는 것"이 가장 위험합니다. 적재 쿼리에서 걸러 내기 전에, **걸러질 행이 몇 개인지 먼저 셉니다.** 0이면 데이터가 깨끗한 것이고, 0이 아니면 왜 그런지 알아야 합니다.

**요구사항**:
- `seoul_metro_transfer_utf8.csv` 를 읽어 아래 두 값을 한 번에 세고, 결과의 첫 행을 **`check`** 에 담으세요.
  - 별칭 **`전체행`**: 전체 행 수
  - 별칭 **`빈이름`**: `역명` 이 비어 있거나 공백만 있는 행 수
- 빈 칸은 `null` 로 들어오므로 `trim(coalesce(row['역명'], ''))` 이 빈 문자열인지로 판정하세요.
- 읽을 파일은 쿼리에 직접 쓰지 말고 **`$path` 파라미터**로 넘기세요(`path='file:///seoul_metro_transfer_utf8.csv'`). 쿼리 문자열은 **`check_query`** 에 담아 두세요.

**예시**: `check` 는 `{'전체행': 73, '빈이름': 0}` 입니다. 이 데이터는 깨끗합니다.

> 채점 셀은 같은 `check_query` 를 **CP949 원본**(`seoul_metro_transfer_cp949.csv`)으로 한 번 더 실행합니다. 점검 쿼리는 대상 파일만 갈아 끼워 재사용하는 것이 정상입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 집계 함수 하나로 전체를 세고, 조건에 맞는 것만 세는 표현을 하나 더 둔다. 파일 이름은 파라미터로 받는다.

세부구현:
1. LOAD CSV WITH HEADERS 로 파라미터가 가리키는 파일을 읽는다.
2. RETURN 절에서 전체 행을 세는 집계와, 조건이 참일 때만 세는 집계를 나란히 쓴다.
   2-1. 조건부로 세려면 CASE WHEN 조건 THEN 1 END 를 세는 함수 안에 넣는다.
3. 결과의 첫 행을 check 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert '전체행' in check and '빈이름' in check, \
    f'별칭을 전체행·빈이름 으로 맞춰야 합니다(현재 키: {list(check)})'
assert check['전체행'] == 73, \
    f'전체행은 73 이어야 합니다(현재 {check["전체행"]}). 별칭을 전체행 으로 맞췄는지 확인하세요'
assert check['빈이름'] == 0, \
    f'이 데이터에는 빈 역명이 없습니다(현재 {check["빈이름"]}). 조건식을 확인하세요'
# 같은 점검을 CP949 원본에 돌려 본다. 머리글이 깨져 역명 칸을 못 찾으므로 전 행이 '빈이름' 으로 잡힌다
dirty = run_cypher(check_query, path='file:///seoul_metro_transfer_cp949.csv')[0]
assert (dirty['전체행'], dirty['빈이름']) == (73, 73), \
    f'CP949 원본은 73행 모두 빈이름 으로 나와야 합니다(현재 {dirty}). 파일 이름을 쿼리에 박지 말고 $path 로 받으세요'
print('✅ 통과!')

## 4. LOAD CSV 로 형변환 적재하기
**배경**: CSV 의 값은 **전부 문자열**입니다. 숫자로 비교·합산하려면 적재할 때 변환해야 합니다.

**요구사항**:
- `seoul_metro_transfer_utf8.csv` 를 읽어 `:Station` 노드를 적재하세요. 이 단원의 도구인 **`LOAD CSV WITH HEADERS`** 로 푸는 것을 권합니다. 파이썬으로 읽어 `UNWIND` 로 넣어도 됩니다(채점은 **그래프에 남은 결과**만 봅니다).
  - `MERGE` 의 식별 조건은 **`name`**(`역명`)
  - `SET` 으로 **`weekday`**(`평일(일평균)`)·**`saturday`**(`토요일`)·**`sunday`**(`일요일`)를 **`toInteger`** 로 변환해 채웁니다.
- 적재 후 `Station` 노드 수를 **`n_station`** 에 담으세요.

**예시**: `n_station` 은 **73** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 읽어 행마다 노드를 MERGE 하고, 숫자 칸은 정수로 바꿔 SET 한다. 그다음 개수를 센다.

세부구현:
1. LOAD CSV WITH HEADERS FROM 'file:///파일명' AS row 로 읽는다.
2. 식별 속성만 MERGE 의 중괄호에 넣는다. 머리글이 한글이라 대괄호로 꺼낸다.
3. 나머지 세 값은 SET 으로 채우되 정수 변환 함수를 씌운다.
4. 적재가 끝나면 Station 을 세어 n_station 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_station == 73, \
    f'역은 73개여야 합니다(현재 {n_station}). MERGE 의 식별 속성이 name 인지 확인하세요'
top = run_cypher("MATCH (s:Station {name: '신도림'}) "
                 "RETURN s.weekday AS w, s.saturday AS sa, s.sunday AS su")[0]
assert top['w'] == 269275, \
    f'신도림 의 weekday 는 269275 여야 합니다(현재 {top["w"]}). toInteger 를 썼는지 확인하세요'
assert isinstance(top['sa'], int) and isinstance(top['su'], int), \
    'saturday·sunday 도 정수여야 합니다. 세 칸 모두 toInteger 로 변환했는지 확인하세요'
print('✅ 통과!')

## 5. 멱등성 확인: 다시 적재해도 그대로
**배경**: 적재 스크립트는 여러 번 실행되기 마련입니다. `MERGE` + `SET` 으로 짜 두면 **다시 적재해도 노드가 늘지 않고, 값은 원본대로 되돌아옵니다.** 이번엔 값을 일부러 망가뜨린 뒤 그것을 확인합니다.

**아래 제공 셀이 모든 역의 평일 인원을 0 으로 망가뜨립니다.** 그다음 문제를 푸세요.

**요구사항**:
- **4번과 똑같은 적재**를 한 번 더 실행한 뒤, 전체 `Station` 수를 **`n_again`** 에 담으세요.

**예시**: `n_again` 은 여전히 **73** 이고, 평일 인원 합계가 **4,814,846** 로, 신도림 의 `weekday` 가 **269,275** 로 되돌아옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4번의 적재를 그대로 한 번 더 실행하고, 역 수를 다시 센다. SET 이 값을 원본으로 다시 덮어쓴다.

세부구현:
1. 4번과 똑같은 적재 쿼리를 다시 실행한다.
2. Station 노드 수를 세어 n_again 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 모든 역의 평일 인원을 0 으로 망가뜨립니다. 이 셀은 실행만 하세요.
# (한 곳만 망가뜨리면 그 한 값만 손으로 되돌려도 다음 채점을 통과해 버립니다)
run_cypher("MATCH (s:Station) SET s.weekday = 0")
broken = run_cypher("MATCH (s:Station) "
                    "RETURN count(s) AS n, sum(s.weekday) AS w")[0]
print(f"평일 인원 합계를 {broken['w']} 으로 바꿨습니다(대상 {broken['n']}개).")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_again == 73, \
    f'재적재해도 73개여야 합니다(현재 {n_again}). 4번과 같은 적재 쿼리를 그대로 다시 실행했는지 확인하세요'
# 한 값만 손으로 되돌려도 통과하지 않게 전 역의 합계를 본다
total_weekday = run_cypher("MATCH (s:Station) RETURN sum(s.weekday) AS w")[0]['w']
assert total_weekday == 4814846, \
    f'평일 인원 합계가 {total_weekday} 입니다. 같은 적재를 한 번 더 돌려 값을 되돌리세요'
restored = run_cypher("MATCH (s:Station {name: '신도림'}) RETURN s.weekday AS w")[0]['w']
assert restored == 269275, \
    f'신도림 의 weekday 가 269275 로 되돌아와야 합니다(현재 {restored}). 4번과 똑같은 적재를 다시 실행했는지 확인하세요'
print('✅ 통과!')

## 6. 파라미터로 조회하기 1): 기준 이상 세기
**배경**: "평일 환승인원이 10만 이상인 역이 몇 개?" 같은 물음은, 기준값을 **파라미터**로 넘겨 조회합니다. 값만 바꿔 같은 쿼리를 재사용할 수 있습니다.

**요구사항**:
- `weekday` 가 **`$threshold` 이상**인 `Station` 의 개수를 세는 쿼리를 **`busy_query`** 에 담고, `run_cypher(busy_query, threshold=100000)` 로 실행해 개수를 **`n_busy`** 에 담으세요.
- 쿼리 안에는 값을 직접 쓰지 말고 **`$threshold`** 를 쓰세요. 개수의 별칭은 **`n`** 으로 합니다.

**예시**: 평일 100,000명 이상인 역은 **12개** 입니다.

> 채점 셀은 같은 `busy_query` 를 **다른 기준값**으로 한 번 더 실행합니다. 기준값만 바꿔 같은 쿼리를 재사용할 수 있다는 것이 파라미터를 쓰는 이유입니다.

<details><summary>힌트</summary>

```text
접근방법:
- weekday 를 파라미터와 비교해 거른 뒤 개수를 센다. 값은 호출할 때 인자로 넘긴다.

세부구현:
1. Station 을 잡아 weekday 가 파라미터 이상인 것만 WHERE 로 거르는 쿼리를 문자열로 만들어 담는다.
2. 거른 개수를 세어 지문에 적힌 별칭으로 반환한다.
3. 그 문자열과 함께 threshold 를 인자로 넘겨 실행하고, 결과의 개수를 n_busy 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_busy == 12, \
    f'12개여야 합니다(현재 {n_busy}). 초과(>)가 아니라 이상(>=)인지, toInteger 로 적재했는지 확인하세요'
# 기준값만 바꿔 같은 쿼리를 다시 돌린다. 값을 손으로 적어 넣은 답안은 여기서 걸린다
_row = run_cypher(busy_query, threshold=200000)[0]
assert 'n' in _row, \
    f'개수의 별칭은 n 이어야 합니다(현재 키: {list(_row)})'
higher = _row['n']
assert higher == 2, \
    f'기준을 200,000 로 올리면 2개여야 합니다(현재 {higher}). 값을 직접 적지 말고 $threshold 를 쓴 쿼리를 busy_query 에 담으세요'
print('✅ 통과!')

## 7. 파라미터로 조회하기 2): 이름으로 값 꺼내기
**배경**: 특정 역의 값을 꺼낼 때도 이름을 **파라미터**로 넘깁니다.

**요구사항**:
- `$name` 으로 넘긴 이름의 역의 **`saturday`** 값을 돌려주는 쿼리를 **`sat_query`** 에 담고, `run_cypher(sat_query, name='왕십리')` 로 실행해 값을 **`sat`** 에 담으세요. 반환 별칭도 **`sat`** 으로 합니다.

**예시**: `왕십리` 의 토요일 환승인원은 **138,662** 입니다.

> 채점 셀은 같은 `sat_query` 를 **다른 역 이름**으로 한 번 더 실행합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 이름을 파라미터로 받아 그 역 하나를 잡고, 토요일 값을 반환한다.

세부구현:
1. MATCH 의 중괄호에 name 조건을 파라미터로 둔 쿼리를 문자열로 만들어 담는다.
2. saturday 를 지문에 적힌 별칭과 함께 반환한다.
3. 그 문자열과 함께 name 인자를 넘겨 실행하고, 결과 첫 행의 값을 sat 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sat == 138662, \
    f'138662 여야 합니다(현재 {sat}). saturday 속성을 꺼냈는지, 이름을 파라미터로 넘겼는지 확인하세요'
# 이름만 바꿔 같은 쿼리를 다시 돌린다. 값을 손으로 적어 넣은 답안은 여기서 걸린다
_row = run_cypher(sat_query, name='고속터미널')[0]
assert 'sat' in _row, \
    f'반환 별칭은 sat 이어야 합니다(현재 키: {list(_row)})'
other_sat = _row['sat']
assert other_sat == 161327, \
    f'고속터미널 의 토요일은 161327 입니다(현재 {other_sat}). 값을 직접 적지 말고 $name 을 쓴 쿼리를 sat_query 에 담으세요'
print('✅ 통과!')

## 8. apoc.meta.stats 로 스키마 훑기
**배경**: 적재가 끝나면 "그래프에 지금 무엇이 얼마나 있나"를 한눈에 확인합니다.

**요구사항**:
- `CALL apoc.meta.stats() YIELD labels RETURN labels` 를 **`stats_query`** 에 담고, 그것을 실행해 **레이블별 노드 수** 사전을 **`labels`** 에 담으세요.

**예시**: `labels` 는 `{'Station': 73}` 입니다.

> 채점 셀은 임시 노드(`:TryStation`) 하나를 만든 뒤 같은 `stats_query` 를 다시 실행하고, 확인이 끝나면 그 노드를 지웁니다. 적재 검증에 쓰는 통계라 **그때그때 다시 세어야** 쓸모가 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- APOC 의 스키마 통계 프로시저를 호출해 레이블별 개수만 받아 온다.

세부구현:
1. CALL 로 프로시저를 부르고 YIELD 로 필요한 항목만 받는 쿼리를 문자열로 담는다.
2. RETURN 으로 그 항목을 돌려준다.
3. 그 쿼리를 실행해 결과 첫 행에서 값을 꺼내 labels 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert labels == {'Station': 73}, \
    f'{{"Station": 73}} 이어야 합니다(현재 {labels}). 다른 레이블을 만들었다면 맨 위 초기화 셀부터 이 문항까지 차례로 다시 실행하세요'
# 임시 노드를 하나 만들고 같은 쿼리를 다시 돌린다. 값을 손으로 적어 넣은 답안은 여기서 걸린다
run_cypher("CREATE (:TryStation)")
try:
    again = run_cypher(stats_query)[0]['labels']
finally:
    # 위에서 실패해도 임시 노드는 반드시 지운다. 남으면 뒤 문제의 건수가 조용히 어긋난다
    run_cypher("MATCH (n:TryStation) DELETE n")
assert again == {'Station': 73, 'TryStation': 1}, \
    f'임시 노드까지 세면 {{"Station": 73, "TryStation": 1}} 이어야 합니다(현재 {again}). 값을 직접 적지 말고 stats_query 를 실행한 결과를 담으세요'
print('✅ 통과!')

## 9. UNWIND 로 소량 추가 적재
**배경**: 파일이 아니라 **파이썬 리스트**로 들고 있는 몇 건을 넣을 때는 `UNWIND` 를 씁니다. 리스트를 파라미터로 넘기면 한 번의 왕복으로 끝납니다(교안_01 2-2).

**요구사항**:
- 아래 제공된 `extra` 리스트(환승역이 아니라 이 파일에 없는 실제 역 2곳)를 `$rows` 로 넘겨 `UNWIND` 로 `:Station` 노드를 적재하세요(`MERGE` 의 식별 조건은 `name`, 인원 값은 넣지 않습니다).
- 적재 쿼리는 **`extra_q`** 변수에 담고 `run_cypher(extra_q, rows=extra)` 로 실행하세요(자가채점이 그 쿼리가 정말 `UNWIND` 와 `$rows` 를 쓰는지 봅니다).
- 적재 후 전체 `Station` 수를 **`n_after`** 에 담으세요.

**예시**: `n_after` 는 **75** 입니다(73 + 2).

<details><summary>힌트</summary>

```text
접근방법:
- 리스트를 통째로 파라미터로 넘기고, 쿼리 안에서 한 줄씩 펼쳐 노드를 만든다.

세부구현:
1. UNWIND $rows AS row 로 리스트를 행으로 펼치는 쿼리를 문자열로 만들어 extra_q 에 담는다.
2. 펼친 각 행의 name 을 식별 조건으로 노드를 MERGE 한다.
3. run_cypher(extra_q, rows=extra) 로 실행한다.
4. Station 수를 다시 세어 n_after 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 추가할 역 목록입니다. 이 셀은 실행만 하세요.
extra = [{'name': '을지로입구'}, {'name': '종각'}]
print(extra)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_after == 75, \
    f'75개여야 합니다(현재 {n_after}). rows=extra 를 넘겼는지, MERGE 를 썼는지 확인하세요'
blank = run_cypher("MATCH (s:Station) WHERE s.weekday IS NULL RETURN count(s) AS n")[0]['n']
assert blank == 2, \
    f'인원 값이 없는 역이 2곳이어야 합니다(현재 {blank}). 추가한 역에는 인원을 넣지 않습니다'
# 손으로 두 줄 적어도 역 수는 맞는다. 그래서 쿼리 자체가 UNWIND 와 파라미터를 쓰는지 본다
assert 'UNWIND' in extra_q.upper(), \
    'extra_q 에 UNWIND 로 리스트를 펼치는 쿼리를 담으세요. 역 이름을 한 줄씩 손으로 적는 방식이 아닙니다'
# 글자만 맞으면 통과하지 않게, 두 역을 지우고 학생 쿼리를 채점이 직접 다시 돌린다
run_cypher("MATCH (s:Station) WHERE s.name IN $names DETACH DELETE s",
           names=[r['name'] for r in extra])
run_cypher(extra_q, rows=extra)
_after = run_cypher("MATCH (s:Station) RETURN count(s) AS n")[0]['n']
assert _after == 75, \
    f'extra_q 를 다시 돌렸더니 역이 {_after}곳입니다. 그 쿼리가 실제로 적재하는지 확인하세요'
assert '$rows' in extra_q, \
    '리스트는 $rows 파라미터로 넘겨야 합니다. 역 이름을 쿼리 문자열에 직접 적지 마세요'
print('✅ 통과!')

## 10. 배치 트랜잭션으로 파생 값 채우기
**배경**: 수만 행을 한 트랜잭션에 쥐고 있으면 메모리가 터집니다. `CALL (row) { ... } IN TRANSACTIONS` 로 끊어 커밋합니다. 이 파일은 작지만 **문법을 손에 익히는 것**이 목적입니다.

**요구사항**:
- `seoul_metro_transfer_utf8.csv` 를 다시 읽어, 각 역의 세 요일 인원을 더한 값을 **`total`** 속성에 채우세요.
  - **`CALL (row) { ... } IN TRANSACTIONS OF 20 ROWS`** 를 쓰세요.
  - 중괄호 안에서는 `MATCH (s:Station {name: row['역명']})` 로 **이미 있는 노드를 찾아** `SET` 합니다.
- 적재 쿼리는 **`total_q`** 변수에 담고 `run_cypher(total_q)` 로 실행하세요(자가채점이 그 쿼리가 정말 배치 블록을 쓰는지 봅니다).
- 채운 뒤 `total` 이 비어 있지 않은 `Station` 수를 **`n_total`** 에 담으세요.

**예시**: `n_total` 은 **73** 이고, `신도림` 의 `total` 은 **664,139** 입니다(9번에서 추가한 2곳은 파일에 없어 그대로 비어 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 다시 흘려보내되, 각 행 처리를 배치 블록 안에 넣어 일정 행마다 커밋하게 한다.

세부구현:
1. LOAD CSV WITH HEADERS 로 파일을 읽는다.
2. CALL 뒤 괄호에 들여보낼 변수를 적고, 중괄호 안에 한 행에 할 일을 쓴다.
   2-1. 이미 있는 노드이므로 MERGE 가 아니라 MATCH 로 찾는다.
   2-2. 세 칸을 정수로 바꿔 더한 값을 total 에 SET 한다.
3. 중괄호 뒤에 배치 크기를 지정하는 절을 붙인다.
4. 그 쿼리 문자열을 total_q 에 담고 run_cypher(total_q) 로 실행한다.
5. total 이 비어 있지 않은 Station 을 세어 n_total 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 배치 블록을 빼고 한 번에 SET 해도 값은 같다. 그래서 쿼리 자체가 그 문법을 쓰는지 본다
assert 'IN TRANSACTIONS' in total_q.upper(), \
    'total_q 에 CALL (row) { ... } IN TRANSACTIONS 블록을 쓴 쿼리를 담으세요'
import re as _re
assert _re.search(r'IN\s+TRANSACTIONS\s+OF\s+20\s+ROWS', total_q, _re.I), \
    '배치 크기는 20행입니다. IN TRANSACTIONS OF 20 ROWS 로 적었는지 확인하세요'
assert n_total == 73, \
    f'total 을 채운 역이 73개여야 합니다(현재 {n_total}). 파일 73행을 모두 처리했는지 확인하세요'
got = run_cypher("MATCH (s:Station {name: '신도림'}) RETURN s.total AS t")[0]['t']
assert got == 664139, \
    f'신도림 의 total 은 664139 이어야 합니다(현재 {got}). 세 요일을 모두 더했는지 확인하세요'
print('✅ 통과!')

## 11. 자료의 기준일을 날짜 노드로 올리기
**배경**: 지금 그래프에는 이 인원 값이 **언제 기준**인지가 어디에도 없습니다. 나중에 다른 시점 자료가 들어오면 어느 값이 언제 것인지 알 길이 없어집니다. 이 파일은 포털에서 받은 원본 이름이 `서울교통공사_환승역환승인원정보_20251130.csv` 라 **2025-11-30 기준**입니다.

기준일을 역마다 속성으로 복사해 넣을 수도 있지만, 여러 역이 **함께 나눠 갖는 값**이라 노드로 올려 둡니다(교안_02 3절의 판단 기준표, 4절의 날짜 판단). 그러면 "같은 날 조사된 것"이 관계를 따라가는 일이 됩니다.

**요구사항**:
- 기준일을 담을 노드를 하나 만드세요.
  - 레이블은 **`SurveyDay`**, 속성 이름은 **`date`**, 값은 **2025-11-30 을 날짜 자료형으로** 넣습니다(따옴표만 씌운 글자로 넣으면 크기 비교가 에러 없이 빈 결과가 됩니다).
  - **아래 제공 셀이 기준일 노드 하나와 `신도림` 관계 하나를 미리 만들어 둡니다.** 그 위에 적재하는 것이라, 이미 있는 것을 다시 만들지 않는 `MERGE` 로 짜야 노드가 하나로 유지됩니다.
- **인원 값이 들어 있는 역**(`weekday` 가 비어 있지 않은 역)만 그 노드로 이으세요.
  - 관계 타입은 **`SURVEYED_ON`**, 방향은 **역에서 날짜 쪽**입니다.
  - 9번에서 추가한 2곳은 이 파일에 없는 역이라 잇지 않습니다.
- 적재한 뒤 `SurveyDay` 노드 수를 **`n_day`**, `SURVEYED_ON` 관계 수를 **`n_surveyed`** 에 담으세요.

**예시**: `n_day` 는 **1**, `n_surveyed` 는 **73** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 날짜 노드를 하나 만들어 두고, 조건에 맞는 역을 찾아 그 노드로 잇는다.

세부구현:
1. MERGE 로 날짜 노드를 만든다. 날짜는 날짜를 만드는 함수로 감싸 넣는다.
2. 만든 노드를 WITH 로 다음 절에 넘긴다. 넘기지 않으면 아래 MATCH 가 앞 절과 이어지지 않는다.
3. 인원 값이 비어 있지 않은 Station 을 MATCH 한다.
4. 그 역과 날짜 노드를 MERGE 로 잇는다.
5. 날짜 노드 수와 관계 수를 각각 세어 담는다.
```

</details>

In [ ]:
# [제공 코드] 기준일 노드 하나와 신도림 관계 하나를 미리 심어 둡니다. 이 셀은 실행만 하세요.
# 빈 그래프에서 시작하면 CREATE 로 짜도 노드가 하나뿐이라 티가 안 납니다.
# 이미 하나가 있는 상태에서 적재해야 MERGE 와 CREATE 의 차이가 건수로 드러납니다.
run_cypher("MERGE (d:SurveyDay {date: date('2025-11-30')}) "
           "WITH d MATCH (s:Station {name: '신도림'}) "
           "MERGE (s)-[:SURVEYED_ON]->(d)")
print('미리 심어 둔 기준일 노드:', run_cypher("MATCH (d:SurveyDay) RETURN count(d) AS n")[0]['n'])

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 1) 날짜를 글자로 적으면 크기 비교가 에러 없이 빠진다. 그러면 이 수가 노드 수보다 적어진다
typed = run_cypher("MATCH (d:SurveyDay) WHERE d.date >= date('2025-01-01') "
                   "RETURN count(d) AS n")[0]['n']
assert typed == n_day, \
    f'기준일을 날짜 자료형으로 넣어야 합니다(노드 {n_day}개 중 크기 비교에 걸린 것은 {typed}개). '  \
    f'따옴표만 씌운 글자가 아닌지 확인하세요'
assert n_day == 1, \
    f'기준일 노드는 1개여야 합니다(현재 {n_day}). 제공 셀이 미리 만들어 둔 노드를 '  \
    f'다시 만들지 않으려면 CREATE 가 아니라 MERGE 여야 합니다'
# 2) 방향은 건수로 보지 않는다. 건수로 보면 '역을 더 이었다' 와 '거꾸로 이었다' 가
#    서로의 에러 메시지를 가로채 학생이 엉뚱한 데를 고치게 된다. 거꾸로 난 것만 따로 센다
backwards = run_cypher("MATCH (:SurveyDay)-[r:SURVEYED_ON]->(:Station) "
                       "RETURN count(r) AS n")[0]['n']
assert backwards == 0, \
    f'역에서 날짜 쪽으로 이어야 합니다(거꾸로 이어진 관계 {backwards}개). 화살표 방향을 확인하세요'
# 학생 변수만 보면 적재를 안 하고 값만 적어도 통과한다. 그래프에서 다시 센다
_linked = run_cypher("MATCH (:Station)-[r:SURVEYED_ON]->(:SurveyDay) "
                     "RETURN count(r) AS n")[0]['n']
assert _linked == 73, \
    f'그래프의 SURVEYED_ON 관계가 {_linked}건입니다. 73건이어야 합니다'
assert n_surveyed == 73, \
    f'73곳이어야 합니다(현재 {n_surveyed}). 인원 값이 없는 역은 빼고, 관계도 MERGE 로 이어야 합니다'
print('✅ 통과!')